In [ ]:
import pandas as pd
import glob
import networkx as nx
import networkx.algorithms.community as nx_comm
#import bamboolib
import pelote
from ipysigma import Sigma, SigmaGrid
import csv
import matplotlib.pyplot as plt
import numpy as np
import re

In [ ]:
import os
current_directory = os.getcwd()
print("Current directory : ", current_directory)

aymeric =  "/home/aymeric/python-scripts/anses_medialab/datas/" #aymeric
jp = '~/Dropbox/Mac/Desktop/CRD Anses/all3/' # Jean Philippe
jp_index = '~/Dropbox/Mac/Desktop/CRD Anses/code/indexation_results/' # Jean Philippe index

if 'aymeric' in current_directory:
    path_base = aymeric

elif 'Mac' in current_directory:
    path_base = jp
elif 'd:/Projects' in current_directory:
    path_base = "d:/Projects/Medialab/"

print("Path base : ", path_base)

# Chargement des retweets

In [ ]:
# Using '*' pattern 
#print('\nNamed with wildcard *:')
i=-1
dic_id={}
for x in [x for x in pd.read_csv(f"{path_base}data_for_industry/20251017_pesticides_industries_cleaned_version.csv").columns if 'id' in x]:
    dic_id[x]=str
dic_id
users0 = pd.read_csv(f"{path_base}data_for_industry/20251017_pesticides_industries_cleaned_version.csv", sep =",", dtype=dic_id)


In [ ]:
users0.columns

In [ ]:
users0 = users0[['user_id', 'user_screen_name', 
       'user_description', 'account_publication', 'industry', 'Type_entite', 'company', 'UIPP','field_work', 'metier', 'Ingenieur', 'Agronome', 'industry_verif']]

In [ ]:
list_col = ['user_id',  'user_screen_name',
       'user_description', 'account_publication', 'Type_entite']

In [ ]:
# Using '*' pattern 
#print('\nNamed with wildcard *:')
i=-1
dic_id={}
for x in [x for x in pd.read_csv(f'{path_base}the_true_all_annotated_account.csv').columns if 'id' in x]:
    dic_id[x]=str
dic_id
users1 = pd.read_csv(f'{path_base}the_true_all_annotated_account.csv', sep =",", dtype=dic_id)

In [ ]:
users1["user_description"] = users1.user_description.str.replace("\n", " ")
users1["user_description"] = users1.user_description.str.replace(",", " ")
#users1.to_csv(f"{path_base}the_true_all_annotated_account.csv", sep = ",", index = False)

In [ ]:
#users1 = users1.drop_duplicates().loc[users1["platform"]=="twitter"]
users1 = users1[['user_id', 'user_screen_name', 'user_name', 'Type_entite',
       'User_role', 'user_description', 'account_publication', 
       'Synthetic_logics']].drop_duplicates()

In [ ]:
keep_all = True #Pour garder ou pas les comptes d'industries dont on a pas collecté les retweets



if keep_all == True:
    users1b = users1.merge(users0, on = list_col, how = "left")
    already_in_users1 = [x for x in users1b.user_id.unique()]
    users2 = users0.loc[~users0["user_id"].isin(already_in_users1)]
    #print(len(users), len(users0))
    
    users = pd.concat([users1b, users2])
else:
    users = users1.merge(users0, on = list_col, how = "left")
    


len(users)

In [ ]:
users.loc[(users["industry_verif"].isna()), "industry"]=False
users.loc[(users["industry_verif"] == True), "industry"]=True
users.loc[(users["industry"].isna()) | (users["industry"]==False) , "company"]="irrelevent"
users.loc[(users["industry"]==True) & (users["company"].isna()) , "company"]= "Unknown"



users = users.drop_duplicates(["user_screen_name","user_id","industry", "Type_entite"])



In [ ]:
users.industry.value_counts()

## Analyse hashtags

In [ ]:
dic_agrotwit = {}
dic_agribash = {}
users_with_desc = users.loc[~users.user_description.isna()]
print(len(users_with_desc), len(users))
for n, x in enumerate(users_with_desc.user_description):
    if re.search("franceagritwittos|france agritwittos|fragritwittos|fragtw", x.lower()):
        dic_agrotwit[users_with_desc.user_id.iloc[n]] = True
    else:
        dic_agrotwit[users_with_desc.user_id.iloc[n]] = False
    if re.search("agribashing|agri bashing|agri-bashing", x.lower()):
         dic_agribash[users_with_desc.user_id.iloc[n]] = True
    else:
         dic_agribash[users_with_desc.user_id.iloc[n]] = False

users["fragtw_desc"] = users.user_id.map(dic_agrotwit.get)
users["agribashing_desc"] = users.user_id.map(dic_agribash.get)



users.agribashing_desc.value_counts()

In [ ]:
users.Synthetic_logics.value_counts()
users.groupby(["Synthetic_logics", "fragtw_desc"]).agg(nb=("user_id","size")).sort_values(by= ["fragtw_desc","nb"], ascending= [False,False]).reset_index()

In [ ]:
users.columns

In [ ]:
users.loc[(users.fragtw_desc== True), "fragtw"] = True
users.loc[(users.fragtw_desc== False), "fragtw"] = False

users.loc[(users.agribashing_desc== True) , "agribashing"] = True
users.loc[(users.agribashing_desc== False), "agribashing"] = False

In [ ]:
dic_hash = {}
list_hashtags  = []
users_with_desc = users.loc[~users.user_description.isna()]
print(len(users_with_desc), len(users))
for n, x in enumerate(users_with_desc.user_description):
    hashtags = []
    for y in re.split('\s|,\s|\.\s|;\s', x):
        if re.findall("^#.*", y.lower()):
            hashtags.append(y.lower())
            if y.lower() not in list_hashtags:
                list_hashtags.append(y.lower())
    if len(hashtags) > 0:
        dic_hash[users_with_desc.user_id.iloc[n]] = "|".join(hashtags)
    else:
        dic_hash[users_with_desc.user_id.iloc[n]] = np.nan
    
    
print(len(list_hashtags))




In [ ]:
users["hashtags"] = users.user_id.map(dic_hash.get)
users["hash_split"] = users.hashtags.str.split("|")
users_hash = users.explode("hash_split")
users_hash = users_hash.loc[~(users_hash.hash_split.isna())]
indus_hash = users_hash.loc[(users_hash.industry==True)]

In [ ]:
#import spacy
for x in indus_hash.hash_split.unique():
    print(x.replace("#",""))

In [ ]:
indus_hash.groupby(["Type_entite"]).agg(nb=("hash_split","size"))


indus_hash.groupby(["hash_split"]).agg(nb=("user_id","size")).sort_values("nb", ascending=False)#.to_csv(f'{path_base}data_for_industry/hashtags.csv', sep =",", index=True)

In [ ]:
users.user_description.loc[(users.agribashing == True) & (users.industry_verif==True)].iloc[0]#.sort_values("account_publication", ascending = False)

## Chargement données activité

In [ ]:
# Using '*' pattern 
#print('\nNamed with wildcard *:')
i=-1
dic_id={}
for x in [x for x in pd.read_csv(f'{path_base}/data_for_industry/20250829_champ_activite.csv').columns if 'id' in x]:
    dic_id[x]=str
dic_id
field_work = pd.read_csv(f'{path_base}/data_for_industry/20250829_champ_activite.csv', sep =",", dtype=dic_id)



In [ ]:
users.loc[users.user_screen_name=="we_empower_inc"]
users.at[59, "company"] = "We power inc"
users.loc[users.user_screen_name=="we_empower_inc"]

In [ ]:
dict_field_work = dict(zip(field_work.user_id, field_work.champ))
dict_metier = dict(zip(field_work.user_id, field_work.metier))
users["field_work"] = users.user_id.map(dict_field_work.get)
users["metier"] = users.user_id.map(dict_field_work.get)

In [ ]:
users01 = users.loc[(users.industry==True) & (users.company!="Unknown")]
users01 = users01.loc[users01.user_screen_name!="PatrickDucam"]
users01.company.value_counts()

In [ ]:
users01.Type_entite.value_counts()
users01_grouped = users01.groupby(["Type_entite","company"]).agg(nb = ("user_id","size")).reset_index()
pvt_table = users01_grouped.pivot(index='company', columns='Type_entite', values='nb').fillna(0)
pvt_table.to_clipboard()


In [ ]:
users01.groupby(["company","user_screen_name"]).agg(nb = ("user_id","size")).reset_index()
users01.loc[(users01.company=="FMC") & (users01.Type_entite=="Organisation")]

In [ ]:
users01.Type_entite.value_counts()

# Activité des salariés

In [ ]:
users01.loc[(users01.Type_entite=="Personne") & (users01.user_description.str.lower().str.contains("ingé")), "Ingenieur"] = True
users01.loc[(users01.Type_entite=="Personne") & ~(users01.user_description.str.lower().str.contains("ingé")), "Ingenieur"] = False
users01.loc[(users01.Type_entite=="Personne") & (users01.user_description.str.lower().str.contains("agr*")), "Agronome"] = True
users01.loc[(users01.Type_entite=="Personne") & ~(users01.user_description.str.lower().str.contains("agr*")), "Agronome"] = False


users01[["user_screen_name", "user_description", "Ingenieur", "Agronome"]]


In [ ]:
users01.Agronome.value_counts()


In [ ]:
users01.field_work.value_counts()
personal_users= users01.loc[users01.Type_entite=="Personne"]
personal_users_grouped = users01.groupby(["field_work"]).agg(nb = ("user_id","size")).reset_index()
personal_users_grouped.sort_values("nb", ascending=False).to_clipboard()

In [ ]:
u = users01.loc[~users01.metier.isna()]
u.loc[(u.Type_entite=="Personne") & (~(u.metier.str.lower().str.contains("resp|manage|direct"))| ~(u.user_description.str.lower().str.contains("resp|manage|direct"))), "Direction"] = False
u.loc[(u.Type_entite=="Personne") & ((u.metier.str.lower().str.contains("resp|manage|direct"))| (u.user_description.str.lower().str.contains("resp|manage|direct"))), "Direction"] = True
u.Direction.value_counts()

In [ ]:
personal_users= u.loc[u.Type_entite=="Personne"]
personal_users_grouped = u.groupby(["Direction"]).agg(nb = ("user_id","size")).reset_index()
personal_users_grouped.sort_values("nb", ascending=False)#.to_clipboard()

In [ ]:
users01.groupby(["Type_entite","fragtw_desc"]).agg(nb = ("user_id","size")).reset_index()
u_fragtw = u.loc[(u.fragtw_desc == True)]
for x in u_fragtw.user_description.loc[(u_fragtw.Direction==True)]:
    print("######")
    print(x)

In [ ]:
users01["industry_verif"]= True
dict_industry_verif = dict(zip(users01.user_id, users01.industry_verif))
users01.columns

In [ ]:
users01.field_work.unique()

In [ ]:
users0.columns
users0 = users0.merge(users01[['user_id', 'user_screen_name', 'field_work', 'metier', 'Ingenieur',
       'Agronome', 'industry_verif']], on = ["user_id", "user_screen_name"], how = "left")


In [ ]:
users0.columns

In [ ]:
users1.columns
users1 = users1.merge(users01[['user_id', 'user_screen_name', 'industry_verif']], on = ["user_id", "user_screen_name"], how = "left")

In [ ]:
users.columns
users = users.merge(users01[['user_id', 'user_screen_name', 'Ingenieur',
       'Agronome', 'industry_verif']], on = ["user_id", "user_screen_name"], how = "left")

In [ ]:
import glob
tweet_path = f"{path_base}/tweets_pesticides/tweets_pesticides and cie/"
paths=glob.glob(tweet_path)
paths
i=-1
dic_id={}
for x in [x for x in pd.read_csv(glob.glob(f'{tweet_path}*.csv')[0]).columns if 'id' in x]:
    dic_id[x]=str
dic_id
for name in glob.glob(f'{tweet_path}*.csv'):
    print (name)
    i+=1
    df0=pd.read_csv(name, dtype=dic_id, usecols=['id','user_id','local_time', "text"])
    
    if i>0:
        df=pd.concat([df,df0])
    else:
        df=df0



In [ ]:

tweets= df.drop_duplicates()
tweets['date'] = pd.to_datetime(tweets['local_time']).dt.date
#tweets

print (len(tweets))
tweets=tweets.dropna(subset=['text'])
tweets = tweets.reset_index()

tweets = tweets.loc[~(tweets['text'].str.contains('N.H.L.', case=False, regex=False, na=False))]
print (len(tweets))

In [ ]:
users.loc[(users["industry_verif"].isna()), "industry_verif"]=False

tweets = tweets.merge(users[['user_id', 'user_screen_name', 'Type_entite', 'User_role',
       'Synthetic_logics',
       'company', 'UIPP', 'fragtw_desc', 'agribashing_desc',
       'fragtw', 'agribashing', 'field_work', 'metier', 'Ingenieur',
       'Agronome', 'industry_verif']], on = ["user_id"], how="left")

tweets.industry_verif.value_counts()

In [ ]:
tweets.loc[(tweets["industry_verif"].isna()), "industry_verif"]=False
tweets.industry_verif.value_counts()

# Compter les hasthag

In [ ]:
dic_agrotwit = {}
dic_agribash = {}
tweets_indus = tweets.loc[(tweets["industry_verif"] == True)]
print(len(tweets_indus), len(tweets))
for n, x in enumerate(tweets_indus.text):
    if re.search("fragtw", x.lower()):
        #print("########\n", x.lower())
        dic_agrotwit[tweets_indus.id.iloc[n]] = True
    else:
        dic_agrotwit[tweets_indus.id.iloc[n]] = False
    if re.search("agribashing|agri bashing|agri-bashing", x.lower()):
         dic_agribash[tweets_indus.id.iloc[n]] = True
    else:
         dic_agribash[tweets_indus.id.iloc[n]] = False

tweets_indus["fragtw"] = tweets_indus.id.map(dic_agrotwit.get)
tweets_indus["agribashing"] = tweets_indus.id.map(dic_agribash.get)



tweets_indus.fragtw.value_counts()

In [ ]:
first_users = users.loc[(users.industry_verif==True)].sort_values("account_publication", ascending=False).reset_index()
first_users["rank"] = first_users.index
dict_rank = dict(zip(first_users.user_screen_name, first_users["rank"]))

In [ ]:
user_fragtw = tweets_indus.loc[tweets_indus.fragtw==True].groupby(["user_screen_name"]).agg(nb =("id","size")).sort_values("nb", ascending=False).reset_index()
user_fragtw["rank"]= user_fragtw.user_screen_name.map(dict_rank.get)
user_fragtw.merge(users[["user_screen_name","account_publication", "company", "fragtw_desc"]], on = ["user_screen_name"], how= "left")

In [ ]:
user_agri = tweets_indus.loc[tweets_indus.agribashing==True].groupby(["user_screen_name"]).agg(nb =("id","size")).sort_values("nb", ascending=False).reset_index()
user_agri["rank"]= user_agri.user_screen_name.map(dict_rank.get)
user_agri.merge(users[["user_screen_name","account_publication", "company", "agribashing_desc"]], on = ["user_screen_name"], how= "left")
#np.sum(user_agri.nb)

In [ ]:
user_countermov = user_agri.merge(user_fragtw, on =["user_screen_name"], how= "left").merge(users[["user_screen_name","account_publication", "company","fragtw_desc", "agribashing_desc"]], on = ["user_screen_name"], how= "left").fillna(0)

In [ ]:
plcr = tweets_indus.loc[(tweets_indus.user_screen_name=="sieclevert")]
print(len(tweets_indus), len(tweets))
for n, x in enumerate(plcr.text):
    if re.search("agribashing", x.lower()):
        print("########\n", x.lower(), plcr.date.iloc[n])
    else:
         pass

In [ ]:
sns.scatterplot(data= user_countermov, x= 'nb_x', y='nb_y', hue = "company", size="account_publication")

In [ ]:
users.loc[(users.fragtw_desc == True) & (users.industry_verif==True)].company.value_counts()#.iloc[0]#.sort_values("account_publication", ascending = False)

In [ ]:
tweets.loc[tweets.industry_verif==True]
nb_user = tweets[['user_id','industry_verif']].drop_duplicates()
(nb_user.industry_verif.value_counts()/len(nb_user))*100
(tweets.industry_verif.value_counts()/len(tweets))*100

In [ ]:
indus_tweets = tweets.loc[tweets.industry_verif==True]
indus_grouped = indus_tweets.groupby(["user_screen_name","user_id"]).agg(nb=("id", "size")).sort_values("nb", ascending=True).reset_index()
np.sum(indus_grouped.nb.loc[indus_grouped.nb>=np.quantile(indus_grouped.nb, 0.75)])/np.sum(indus_grouped.nb)
np.quantile(indus_grouped.nb, 0.75)

In [ ]:
i = indus_grouped.sort_values("nb", ascending=False)
np.sum(i.nb.iloc[0])/np.sum(i.nb)
i.iloc[2:6]

In [ ]:
np.sum(i.nb.iloc[2:6])#/np.sum(i.nb)

In [ ]:
import seaborn as sns

In [ ]:
fig, ax = plt.subplots( figsize=(15,12))

sns.barplot(data=indus_grouped, x="user_screen_name", y="nb")

In [ ]:
x = [y for y in indus_grouped.nb]

print(np.min(x))
print(np.max(x))

print("Le nombre d\'occurrences de 1 est", x.count(1))

print("Moyenne : ",np.mean(x)) 
print("Ecart-type : ",np.std(x))

print("Médiane : ", np.median(x))
print("Quantile 0.25 : ", np.quantile(x,0.25))
print("Quantile 0.75 : ", np.quantile(x,0.75))
print("Quantile 0.90 : ", np.quantile(x,0.90))

In [ ]:
indus_grouped.nb.cumsum()

In [ ]:
x = np.array(x)
modalites, effectifs = np.unique(x, return_counts=True)
print("Modalités :", modalites)
print("Effectifs :", effectifs)

In [ ]:
tab_freq = indus_grouped.groupby(["nb"]).agg(n = ("user_id", "size")).sort_values("nb", ascending=True).reset_index()
tab_freq.to_clipboard()

In [ ]:
users01.sort_values("account_publication", ascending=False).head(6)

In [ ]:
indus_tweets = tweets.loc[tweets.industry_verif==True]
company_grouped_tweets = indus_tweets.groupby(["company"]).agg(nb=("id", "size")).sort_values("nb", ascending=False).reset_index()
basf = users01.loc[users01.user_screen_name.isin(["fastier_antony","qualidon", "petatJM","LCJAMET","Pat2816"])].sort_values("account_publication", ascending=False)
print(basf.company.value_counts())
print(np.sum(basf.account_publication))
company_grouped_users = users01.groupby(["company"]).agg(nb=("user_id", "size")).sort_values("nb", ascending=False).reset_index()
company_grouped_users["tweets"] = company_grouped_users.company.map(dict(zip(company_grouped_tweets.company, company_grouped_tweets.nb)).get)
company_grouped_users.to_clipboard()

In [ ]:
indus_tweets.user_screen_name.loc[indus_tweets.Type_entite=="Organisation"].unique()

In [ ]:
u = users.sort_values('account_publication', ascending=False).reset_index()
u.loc[u.industry == True]

In [ ]:
indus_tweets['date'] = pd.to_datetime(pd.to_datetime(indus_tweets['local_time']).dt.date)
indus_tweets['Year'] = indus_tweets['date'].dt.year

In [ ]:
def tracer_graphique(data, variable, d,  title, save_fig=None):
    scale = {"Année":"YE","Mois":"ME","Jour":"d"}
    df1 = data.groupby([variable, "date"]).agg(nb=('id','size')).reset_index()
    fig,ax=plt.subplots(1,figsize=(10, 8))

    #df["num"] = 1 # valeur par article pour comptage
    leg = []
    
    if variable in data.columns:
        for i, j in data.groupby(variable):
            leg.append(i)
            j.set_index("date")["id"].resample(scale[d]).size().plot(ax=ax,style="-")
        plt.legend(leg)
    else:
        #data.set_index("date")["id"].resample(scale[d]).size().plot(ax=ax,style=".-")
        plt.plot(df1.local_time, df1.nb)
    plt.title(f"Distribution par {d.lower()} du corpus {title}")
    #plt.xlabel("Temps (par %s)"%d)
    plt.ylabel("Nombre de tweets")
    plt.tight_layout()
    
    if save_fig is not None:
        plt.savefig(f"{save_fig}.pdf", dpi= 300)
        fig.savefig(f"{save_fig}.jpeg", dpi=100, format = "jpeg")
    else:
        pass
    
    return fig

In [ ]:
def tracer_multi_graphique(data, variable, d,  title, save_fig=None, n_axe = 0):
    scale = {"Année":"YE","Mois":"ME","Jour":"d"}
    df1 = data.groupby([variable, "date"]).agg(nb=('id','size')).reset_index()
    #df["num"] = 1 # valeur par article pour comptage
    leg = []
    
    if variable in data.columns:
        for i, j in data.groupby(variable):
            leg.append(i)
            j.set_index("date")["id"].resample(scale[d]).size().plot(ax=ax[n_axe],style="-")
        ax[n_axe].legend(leg)
        ax[n_axe].set_title(f"Distribution par {d.lower()} des tweets de {title}")
    else:
        #data.set_index("date")["id"].resample(scale[d]).size().plot(ax=ax,style=".-")
        ax[n_axe].plot(df1.local_time, df1.nb)
    #plt.suptitle(f"Distribution par {d.lower()} des tweets de {title}")
    #plt.xlabel("Temps (par %s)"%d)
    plt.ylabel("Nombre de tweets")
    plt.tight_layout()
    
    if save_fig is not None:
        plt.savefig(f"{save_fig}.pdf", dpi= 300)
        fig.savefig(f"{save_fig}.jpeg", dpi=100, format = "jpeg")
    else:
        pass
    
    return fig

In [ ]:
indus_tweets.columns

In [ ]:

df2 = indus_tweets.loc[((indus_tweets["date"]>='2010-06-08 00:00:00') & (indus_tweets["date"]<='2021-09-10 00:00:00'))  ]
corpus = "industries pesticides"
fig = tracer_graphique(df2, variable = "Type_entite", d = "Mois", title = corpus, save_fig=None)

In [ ]:
liste_company = ["bayer","basf","uipp","syngenta", "FMC","de sangosse"]
fig,ax=plt.subplots(len(liste_company),figsize=(15, 30))

for n, c in enumerate(liste_company):
    df2 = indus_tweets.loc[((indus_tweets["date"]>='2010-06-08 00:00:00') & (indus_tweets["date"]<='2021-09-10 00:00:00')) & (indus_tweets.company.isin([c]))]
    corpus = c
    fig = tracer_multi_graphique(df2, variable = "Type_entite", d = "Mois", title = corpus, save_fig=None, n_axe=n)
    

In [ ]:
fig,ax=plt.subplots(6,figsize=(15, 30))
variable= "Type_entite"
scale = {"Année":"YE","Mois":"ME","Jour":"d"}

for n, c in enumerate(["bayer","basf","uipp","syngenta", "FMC","de sangosse"]):
    df2 = indus_tweets.loc[((indus_tweets["date"]>='2010-06-08 00:00:00') & (indus_tweets["date"]<='2021-09-10 00:00:00')) & (indus_tweets.company.isin([c])) ]
    df2b = df2.groupby([variable, "date"]).agg(nb=('id','size')).reset_index()
    leg = []
    if variable in df2.columns:
        for i, j in df2.groupby(variable):
            leg.append(i)
            j.set_index("date")["id"].resample(scale["Mois"]).size().plot(ax=ax[n],style="-")
        ax[n].legend(leg)
        ax[n].set_title("c")


In [ ]:
indus_tweets.company

In [ ]:

df2 = indus_tweets.loc[(indus_tweets["date"]>='2010-06-08 00:00:00') & (indus_tweets["date"]<='2021-09-10 00:00:00')]
df3 = df2.loc[(df2.company.str.contains("basf")) & (df2.user_screen_name!="BonjourSyngenta")]
corpus = "industries pesticides (sans UIPP)"
fig = tracer_graphique(df3, variable = "Type_entite", d = "Année", title = corpus, save_fig=None)

In [ ]:
tweets['date'] = pd.to_datetime(pd.to_datetime(indus_tweets['local_time']).dt.date)
tweets['Year'] = indus_tweets['date'].dt.year



df3 = tweets.loc[(tweets["date"]>='2010-06-08 00:00:00') & (tweets["date"]<='2021-09-10 00:00:00')]
titre = "Distribution mensuelle du corpus original"
fig = tracer_graphique(df3, variable = "industry_verif", d = "Mois", title = titre, save_fig=None)

In [ ]:
tweets.industry_verif.unique()

In [ ]:
tweets.loc[tweets.industry_verif==True, "user_status"] = "Industry"
tweets.loc[tweets.industry_verif==False, "user_status"] = tweets.Synthetic_logics

tweets1 = tweets.loc[~tweets.user_status.isna()]
tweets1['date'] = pd.to_datetime(pd.to_datetime(tweets1['local_time']).dt.date)
tweets1['Year'] = tweets1['date'].dt.year



df3 = tweets1.loc[(tweets1["date"]>='2010-06-08 00:00:00') & (tweets1["date"]<='2021-09-10 00:00:00')]



In [ ]:
df3.user_status.unique()

In [ ]:
df3.loc[(df3.user_status.isin(['Pesticide_free_agriculture','Ecological_perspectives'])), "user_status2"] = "Pesticide free mvt"
df3.loc[(df3.user_status.isin(["Agroindustrial_perspectives","Rationalist_perspectives"])), "user_status2"] = "Pro Pesticide mvt"
df3.loc[(df3.user_status.isin(["Industry"])), "user_status2"] = "Industry"

In [ ]:
path_file = "/home/aymeric/python-scripts/anses_medialab/industries/image/20250902_temporalite"

df4 = df3.loc[df3.user_status.isin(["Industry","Agroindustrial_perspectives",'Pesticide_free_agriculture',"Rationalist_perspectives"])]


titre = "Distribution mensuelle du corpus original"
fig = tracer_graphique(df4, variable = "user_status", d = "Mois", title = titre, save_fig=path_file)

In [ ]:
company_grouped_tweets = indus_tweets.groupby(["company","Type_entite"]).agg(nb=("id", "size")).sort_values("nb", ascending=False).reset_index()
pvt_table = company_grouped_tweets.pivot(index='company', columns='Type_entite', values='nb').fillna(0)
pvt_table.loc[pvt_table.Organisation<pvt_table.Personne, "personnal_user"] = True
pvt_table.loc[pvt_table.Organisation>pvt_table.Personne, "personnal_user"] = False
pvt_table.sort_values("personnal_user").to_clipboard()

In [ ]:
company_grouped_tweets = indus_tweets.loc[indus_tweets.company.str.contains("bayer")].groupby(["Year","Type_entite"]).agg(nb=("id", "size")).sort_values("nb", ascending=False).reset_index()
#pvt_table = company_grouped_tweets.pivot(index='user_screen_name', columns='Type_entite', values='nb').fillna(0)

company_grouped_tweets.sort_values(["Type_entite","Year"], ascending=[True,True])